# Exploración de Features para el Modelo ML
Objetivo: entender los datos disponibles en BigQuery y decidir qué features usar para predecir el resultado de un partido (H/A/D).

In [3]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'credentials.json').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent

CREDENTIALS_FILE = PROJECT_ROOT / 'credentials.json'
if not CREDENTIALS_FILE.is_file():
    raise FileNotFoundError(f'No se encontró credentials.json desde {Path.cwd()}')

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(CREDENTIALS_FILE)
load_dotenv(PROJECT_ROOT / '.env')

client = bigquery.Client()
PROJECT = 'sports-pipeline-506109'
DATASET = 'sports_dbt'

print(f'Conectado a BigQuery usando: {CREDENTIALS_FILE}')

Conectado a BigQuery usando: c:\Users\anaos\Desktop\proyectos\sports-pipeline\credentials.json


## 1. Cargamos los datos de stg_matches

In [4]:
query = f"""
SELECT *
FROM `{PROJECT}.{DATASET}.stg_matches`
"""

df = client.query(query).to_dataframe()
print(f'Filas: {len(df)} | Columnas: {len(df.columns)}')
df.head(5)

c:\Users\anaos\Desktop\proyectos\sports-pipeline\venv\Lib\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Filas: 380 | Columnas: 21


,match_id,matchday,match_date,match_date_only,home_team,away_team,home_score,away_score,total_goals,result,...,stadium,temp_avg,temp_max,temp_min,precipitation,wind_max,weather_code,weather_desc,rain_category,temp_category
0,438524,6,2023-09-24 19:00:00+00:00,2023-09-24,Club Atlético de Madrid,Real Madrid CF,3,1,4,H,...,Cívitas Metropolitano,17.3,25.0,9.6,0.0,10.0,0,Despejado,Seco,Templado
1,438649,18,2023-12-21 18:00:00+00:00,2023-12-21,Cádiz CF,Real Sociedad de Fútbol,0,0,0,D,...,Estadio Nuevo Mirandilla,12.0,16.2,7.8,0.0,26.6,0,Despejado,Seco,Fresco
2,438778,31,2024-04-13 19:00:00+00:00,2024-04-13,Cádiz CF,FC Barcelona,0,1,1,A,...,Estadio Nuevo Mirandilla,19.3,24.8,13.8,0.0,21.8,0,Despejado,Seco,Templado
3,438546,8,2023-10-01 14:15:00+00:00,2023-10-01,Deportivo Alavés,CA Osasuna,0,2,2,A,...,Estadio de Mendizorroza,23.5,32.2,14.7,0.0,9.2,0,Despejado,Seco,Templado
4,438487,2,2023-08-20 17:30:00+00:00,2023-08-20,FC Barcelona,Cádiz CF,2,0,2,H,...,Estadi Olímpic Lluís Companys,28.7,34.9,22.5,0.0,12.9,0,Despejado,Seco,Caluroso


In [5]:
# Distribución de resultados — ¿está balanceado?
print('Distribución de resultados:')
print(df['result'].value_counts())
print()
print(df['result'].value_counts(normalize=True).round(2))

Distribución de resultados:
result
H    167
D    107
A    106
Name: count, dtype: int64

result
H    0.44
D    0.28
A    0.28
Name: proportion, dtype: float64


## 2. ¿El clima afecta al resultado?

In [6]:
# Win rate del equipo local por categoría de lluvia
home_win_by_rain = df.groupby('rain_category').apply(
    lambda x: (x['result'] == 'H').mean()
).round(3)

print('Win rate equipo LOCAL por lluvia:')
print(home_win_by_rain.sort_values(ascending=False))

Win rate equipo LOCAL por lluvia:
rain_category
Lluvia moderada    0.614
Lluvia ligera      0.559
Lluvia fuerte      0.364
Seco               0.362
dtype: float64


In [7]:
# Win rate del equipo local por temperatura
home_win_by_temp = df.groupby('temp_category').apply(
    lambda x: (x['result'] == 'H').mean()
).round(3)

print('Win rate equipo LOCAL por temperatura:')
print(home_win_by_temp.sort_values(ascending=False))

Win rate equipo LOCAL por temperatura:
temp_category
Frío        0.600
Fresco      0.481
Templado    0.416
Caluroso    0.269
dtype: float64


## 3. Features candidatas

In [8]:
# Correlación entre variables numéricas y resultado
# Codificamos result como numérico para ver correlaciones
df['result_num'] = df['result'].map({'H': 1, 'D': 0, 'A': -1})

numeric_cols = ['temp_avg', 'temp_max', 'temp_min', 'precipitation', 'wind_max', 'matchday']
correlaciones = df[numeric_cols + ['result_num']].corr()['result_num'].drop('result_num')

print('Correlación con el resultado (1=local gana, -1=visitante gana):')
print(correlaciones.sort_values(ascending=False).round(3))

Correlación con el resultado (1=local gana, -1=visitante gana):
precipitation    0.023
wind_max        -0.015
matchday        -0.023
temp_min        -0.124
temp_avg        -0.149
temp_max        -0.158
Name: result_num, dtype: float64


In [9]:
# Goles marcados por condición climática
print('Media de goles totales por lluvia:')
print(df.groupby('rain_category')['total_goals'].mean().round(2).sort_values(ascending=False))

Media de goles totales por lluvia:
rain_category
Lluvia moderada    2.95
Lluvia fuerte      2.73
Lluvia ligera      2.67
Seco               2.57
Name: total_goals, dtype: Float64


## 4. Features finales para el modelo

Basándonos en el análisis anterior, estas son las features que usaremos:

**Clima:**
- `temp_avg` — temperatura media
- `precipitation` — lluvia
- `wind_max` — viento

**Partido:**
- `matchday` — jornada (el equipo local puede rendir distinto al principio vs final de temporada)
- `home_team` (codificado) — cada equipo tiene distinto rendimiento como local
- `away_team` (codificado) — cada equipo tiene distinto rendimiento como visitante
- `rain_category` (codificado)
- `temp_category` (codificado)

**Target:** `result` (H / A / D)

In [10]:
# Vista previa del dataset de entrenamiento
feature_cols = ['matchday', 'home_team', 'away_team', 
                'temp_avg', 'precipitation', 'wind_max',
                'rain_category', 'temp_category']
target_col = 'result'

df_ml = df[feature_cols + [target_col]].dropna()
print(f'Dataset ML: {len(df_ml)} filas')
print(f'Nulos: {df_ml.isnull().sum().sum()}')
df_ml.head(10)

Dataset ML: 380 filas
Nulos: 0


,matchday,home_team,away_team,temp_avg,precipitation,wind_max,rain_category,temp_category,result
0,6,Club Atlético de Madrid,Real Madrid CF,17.3,0.0,10.0,Seco,Templado,H
1,18,Cádiz CF,Real Sociedad de Fútbol,12.0,0.0,26.6,Seco,Fresco,D
2,31,Cádiz CF,FC Barcelona,19.3,0.0,21.8,Seco,Templado,A
3,8,Deportivo Alavés,CA Osasuna,23.5,0.0,9.2,Seco,Templado,A
4,2,FC Barcelona,Cádiz CF,28.7,0.0,12.9,Seco,Caluroso,H
5,8,FC Barcelona,Sevilla FC,22.2,0.0,11.6,Seco,Templado,H
6,8,Getafe CF,Villarreal CF,22.2,0.0,9.2,Seco,Templado,D
7,2,Girona FC,Getafe CF,30.0,0.0,13.1,Seco,Caluroso,H
8,8,Girona FC,Real Madrid CF,24.3,0.0,15.3,Seco,Templado,A
9,17,Girona FC,Deportivo Alavés,9.2,0.0,8.2,Seco,Fresco,H


In [11]:
# Guardamos el dataset ML como Parquet para usarlo en el script de entrenamiento
import os
os.makedirs('data/gold', exist_ok=True)
df_ml.to_parquet('data/gold/ml_features.parquet', index=False)
print('Dataset guardado en data/gold/ml_features.parquet')

Dataset guardado en data/gold/ml_features.parquet


Se puede ver que los resultados se obtiene que hay un 44% que ganan de jugar de local, además de que con lluvia se obiene un mejor rendimiento del equipo loval, tanto con lluvia como con frio.